# Morphological and statistical patch analysis

The counterpart of the reference paper's Figure 5: one patch examined in
depth. Ground-truth and predicted surfaces side by side; log-scaled
elevation histograms with maximum-likelihood parametric fits (Normal,
Skew-Normal, Gamma, Log-Normal); empirical variograms with fitted
exponential models; and a metrics block giving standard deviation, effective
range and Jensen-Shannon divergence.

**Patch selection is not by eye.** The patch shown is the validation patch
whose ZNCC is closest to the median of the saved per-patch metrics, so it
is representative rather than a best or worst case. Two further patches --
the 25th and 75th percentile -- are produced alongside it for the appendix.

**What it needs.** `09`'s checkpoint, loaded from disk. One DDIM sampling
pass per patch -- a few seconds each on the GPU. Nothing is retrained.

## GPU configuration

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and training configuration

In [ ]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
REGION = 'tuk'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
SEED = 42
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
LIDAR_SURVEY_DATE = __import__('datetime').date(2024, 4, 16)

# Spatial-block split parameters -- see the split cell below for reasoning
BLOCK_SIZE_M = 1024.0  # 4x the ~256m patch size, generous margin
BUFFER_M = 150.0       # patches within this of a block boundary are dropped entirely

# ---- this notebook: load 09's checkpoint, never train ----
CKPT_PATH = CHECKPOINT_DIR / f's1_{REGION}_pcrtc_realattrs_spatialsplit_unet_best.pth'
METRICS_PATH = OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'
assert CKPT_PATH.exists(), f'checkpoint not found: {CKPT_PATH}'
assert METRICS_PATH.exists(), f'metrics not found: {METRICS_PATH}'

## Import Tessa's baseline implementation

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddpm, p_sample_loop_ddim, p_sample_loop_plms
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

## Sentinel-1/LiDAR dataset adapter -- repeated channels, real attrs (unchanged from 06)

In [ ]:
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])  # 't0' -> 0
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_days = (acq_date - LIDAR_SURVEY_DATE).days
            age_norm = age_days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256), train=False):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.train = train

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Spatial-block split (corrected)

Groups patches into `BLOCK_SIZE_M`-sized geographic blocks by centroid,
shuffles and assigns whole blocks to train/val (never splitting a block),
and drops any patch within `BUFFER_M` of a block boundary entirely --
guaranteeing no train patch and val patch can be adjacent across a block
edge, regardless of how dense the original patch overlap was.

In [ ]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'

def patch_centroid(patch_id):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{patch_id}.tif') as src:
        b = src.bounds
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks = {}
dropped_buffer = []
for pid, (cx, cy) in centroids.items():
    bid, boundary_dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if boundary_dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

kept_total = sum(len(v) for v in blocks.values())
print(f'{len(dropped_buffer)} / {len(paired_ids)} patches dropped as boundary buffer')
print(f'{len(blocks)} spatial blocks remain, containing {kept_total} patches')

block_ids = list(blocks.keys())
random.Random(SEED).shuffle(block_ids)

target_val_patches = int(len(paired_ids) * VAL_FRACTION)  # same target fraction as 06, for comparability
val_ids, train_ids = [], []
running_val_count = 0
for bid in block_ids:
    if running_val_count < target_val_patches:
        val_ids.extend(blocks[bid])
        running_val_count += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'Spatial-block split: train={len(train_ids)}, val={len(val_ids)} '
      f'(target val={target_val_patches}, {len(dropped_buffer)} dropped as buffer)')

train_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, train_ids, CONTEXT_K, TARGET_HW, train=True)
val_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW, train=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

### Verify the split is actually leakage-free

Reuses `pcrtc/08`'s spatial-overlap check directly, applied to this new
split. **This must print `0 / N` before continuing** -- the assertion
below will stop the notebook otherwise, rather than silently training on
a still-contaminated split. If it fails, increase `BLOCK_SIZE_M` or
`BUFFER_M` above and re-run from the split cell.

In [ ]:
from shapely.geometry import box
from shapely.strtree import STRtree

train_boxes = []
for pid in train_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        train_boxes.append(box(*src.bounds))

val_boxes = []
for pid in val_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        val_boxes.append((pid, box(*src.bounds)))

tree = STRtree(train_boxes)
overlap_count = 0
for pid, vbox in val_boxes:
    hits = tree.query(vbox)
    real_overlaps = [h for h in hits if train_boxes[h].intersects(vbox) and not train_boxes[h].touches(vbox)]
    if real_overlaps:
        overlap_count += 1
        print(f'Val patch {pid} still overlaps {len(real_overlaps)} training patch(es)')

print(f'\n{overlap_count} / {len(val_ids)} validation patches overlap a training patch (must be 0)')
assert overlap_count == 0, (
    'Spatial-block split still has leakage -- increase BLOCK_SIZE_M or BUFFER_M '
    'in the config cell above and re-run from the split cell.'
)
print('Confirmed: spatial-block split is leakage-free. Safe to proceed to training.')

split_check_result = {'n_train': len(train_ids), 'n_val': len(val_ids),
                       'n_dropped_buffer': len(dropped_buffer), 'n_overlaps_found': overlap_count,
                       'block_size_m': BLOCK_SIZE_M, 'buffer_m': BUFFER_M}
with (OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_split_check.json').open('w') as f:
    json.dump(split_check_result, f, indent=2)
print('Saved:', OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_split_check.json')

### How much margin, not just pass/fail

The assertion above confirms zero overlap, but that alone doesn't say
whether the split is comfortably clean or just barely passing. This
computes the actual distance from every validation patch to its nearest
training patch -- a healthy result clusters well above `0`, consistent
with `BUFFER_M`; anything close to `0` would mean patches are touching
edge-to-edge with no real separation, worth tightening the buffer for.

In [ ]:
nearest_distances = []
for pid, vbox in val_boxes:
    min_dist = min(vbox.distance(tbox) for tbox in train_boxes)
    nearest_distances.append((pid, min_dist))

nearest_distances_arr = np.array([d for _, d in nearest_distances])
print(f'Distance from each validation patch to its nearest training patch:')
print(f'  min:    {nearest_distances_arr.min():.1f} m')
print(f'  median: {np.median(nearest_distances_arr):.1f} m')
print(f'  mean:   {nearest_distances_arr.mean():.1f} m')
print(f'  max:    {nearest_distances_arr.max():.1f} m')

closest_pid, closest_dist = min(nearest_distances, key=lambda x: x[1])
print(f'\nClosest pair overall: val patch {closest_pid} is {closest_dist:.1f} m from its nearest training patch')
print(f'(buffer was configured at {BUFFER_M} m per side, {2 * BUFFER_M} m total dead zone)')

margin_result = {
    'min_distance_m': float(nearest_distances_arr.min()),
    'median_distance_m': float(np.median(nearest_distances_arr)),
    'mean_distance_m': float(nearest_distances_arr.mean()),
    'max_distance_m': float(nearest_distances_arr.max()),
    'buffer_m_configured': BUFFER_M,
}
with (OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_margin_check.json').open('w') as f:
    json.dump(margin_result, f, indent=2)
print('\nSaved:', OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_margin_check.json')


## Initialize Tessa's ConditionalUNet and scheduler

In [ ]:
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128, embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = (LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Load the trained checkpoint

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded {CKPT_PATH.name}  (epoch {checkpoint.get("epoch")}, val_loss {checkpoint.get("val_loss"):.6f})')

## Choose patches by ZNCC percentile

Median = the representative case shown in the main text. 25th and 75th for
the appendix. Chosen from the saved metrics so the selection is
reproducible and not cherry-picked.

In [ ]:
rows = json.load(METRICS_PATH.open())
rows = [r for r in rows if r['patch_id'] in set(val_ids) and np.isfinite(r['zncc'])]
zs = np.array([r['zncc'] for r in rows])
def pick(q):
    target = np.percentile(zs, q)
    r = min(rows, key=lambda r: abs(r['zncc'] - target))
    return r['patch_id'], r['zncc']
SELECT = {'median': pick(50), 'p25': pick(25), 'p75': pick(75)}
for k, (pid, z) in SELECT.items():
    print(f'{k:<8} patch {pid}   ZNCC {z:.4f}')

## Inference on the selected patches

In [ ]:
sampler = p_sample_loop_ddim
ds = LidarS1Dataset(S1_DIR, LIDAR_DIR, [pid for pid, _ in SELECT.values()], CONTEXT_K, TARGET_HW, train=False)
results = {}
with torch.no_grad():
    for name, (pid, z) in SELECT.items():
        item = ds[[p for p, _ in SELECT.values()].index(pid)]
        target = item['lidar'].unsqueeze(0).to(DEVICE)
        condition = item['s1'].unsqueeze(0).to(DEVICE)
        attrs = item['attrs'].unsqueeze(0).to(DEVICE)
        mask = item['mask'].unsqueeze(0).to(DEVICE).bool()
        pred = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        gt = target[0, 0].cpu().numpy(); pr = pred[0, 0].cpu().numpy(); mk = mask[0].cpu().numpy()
        # demean both over the valid mask, as the metrics do
        gt = gt - gt[mk].mean(); pr = pr - pr[mk].mean()
        results[name] = dict(pid=pid, zncc_saved=z, gt=gt, pred=pr, mask=mk)
        print(f'{name}: patch {pid} sampled')

## Analysis functions

Variogram: isotropic empirical semivariance from random pixel pairs, binned
by lag, with an exponential model $\gamma(h) = c_0 + c\,(1 - e^{-h/a})$
fitted by least squares. Effective range is $3a$, the lag at which the
exponential reaches 95% of its sill.

MLE fits: `scipy.stats` `.fit()`. Gamma and Log-Normal need positive
support, so those two are fitted to values shifted by the sample minimum;
the shift is reported.

In [ ]:
from scipy import stats
from scipy.optimize import curve_fit
from scipy.spatial.distance import jensenshannon

def empirical_variogram(z, mask, n_pairs=200_000, max_lag=100, n_bins=25, seed=0):
    rng = np.random.default_rng(seed)
    ys, xs = np.nonzero(mask)
    i = rng.integers(0, len(ys), n_pairs); j = rng.integers(0, len(ys), n_pairs)
    dy, dx = ys[i] - ys[j], xs[i] - xs[j]
    h = np.hypot(dy, dx); ok = (h > 0) & (h <= max_lag)
    h = h[ok]; g = 0.5 * (z[ys[i], xs[i]] - z[ys[j], xs[j]])[ok] ** 2
    edges = np.linspace(0, max_lag, n_bins + 1); mid = 0.5 * (edges[1:] + edges[:-1])
    which = np.digitize(h, edges) - 1
    sv = np.array([g[which == b].mean() if np.any(which == b) else np.nan for b in range(n_bins)])
    return mid, sv

def exp_model(h, c0, c, a):
    return c0 + c * (1 - np.exp(-h / a))

def fit_variogram(mid, sv):
    ok = np.isfinite(sv)
    p0 = [sv[ok][0] * 0.1, sv[ok].max(), mid[ok][len(mid[ok]) // 3]]
    try:
        popt, _ = curve_fit(exp_model, mid[ok], sv[ok], p0=p0, bounds=([0, 0, 0.1], [np.inf, np.inf, 1e4]), maxfev=20000)
    except Exception:
        popt = [np.nan] * 3
    return popt

def mle_fits(v):
    out = {}
    out['Normal'] = (stats.norm, stats.norm.fit(v), 0.0)
    out['Skew-Normal'] = (stats.skewnorm, stats.skewnorm.fit(v), 0.0)
    shift = v.min() - 1e-3
    out['Gamma'] = (stats.gamma, stats.gamma.fit(v - shift, floc=0), shift)
    out['Log-Normal'] = (stats.lognorm, stats.lognorm.fit(v - shift, floc=0), shift)
    return out

def jsd(a, b, bins=128):
    lo, hi = min(a.min(), b.min()), max(a.max(), b.max())
    pa, _ = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    pb, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    return float(jensenshannon(pa + 1e-12, pb + 1e-12, base=2) ** 2)

## The figure

In [ ]:
from matplotlib.colors import TwoSlopeNorm

def analyse(name, r):
    gt, pr, mk = r['gt'], r['pred'], r['mask']
    gv, pv = gt[mk], pr[mk]
    fig, axes = plt.subplots(3, 2, figsize=(12, 13.5))
    vmax = np.quantile(np.abs(np.concatenate([gv, pv])), 0.995)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    for ax, z, ttl in zip(axes[0], [np.where(mk, gt, np.nan), np.where(mk, pr, np.nan)], ['Input topography (LiDAR, demeaned)', 'Model prediction (demeaned)']):
        im = ax.imshow(z, cmap='RdBu_r', norm=norm); ax.set_title(ttl, fontsize=11, fontweight='bold'); ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label='m')

    # histograms with MLE fits
    for ax, v, ttl in zip(axes[1], [gv, pv], ['LiDAR', 'Prediction']):
        ax.hist(v, bins=120, density=True, color='#C9CFD3', label='data')
        xs = np.linspace(v.min(), v.max(), 400)
        for lab, (dist, params, shift) in mle_fits(v).items():
            ax.plot(xs, dist.pdf(xs - shift, *params), lw=1.4, label=lab)
        ax.set_yscale('log'); ax.set_ylim(bottom=1e-3)
        ax.set_xlabel('elevation residual (m)'); ax.set_ylabel('density (log)')
        ax.set_title(f'{ttl}: distribution and MLE fits', fontsize=10); ax.legend(fontsize=8)

    # variograms
    ax = axes[2, 0]
    summary = {}
    for v2, z, lab, col in [(gv, gt, 'LiDAR', '#1D4E63'), (pv, pr, 'Prediction', '#C2561F')]:
        mid, sv = empirical_variogram(z, mk)
        c0, c, a = fit_variogram(mid, sv)
        ax.plot(mid, sv, 'o', ms=4, color=col, alpha=0.7, label=f'{lab} empirical')
        if np.isfinite(a):
            ax.plot(mid, exp_model(mid, c0, c, a), '-', color=col, lw=1.5, label=f'{lab} exp. fit (range {3*a:.0f} m)')
        summary[lab] = dict(std=float(v2.std()), eff_range=float(3*a) if np.isfinite(a) else np.nan, sill=float(c0 + c) if np.isfinite(c) else np.nan)
    ax.set_xlabel('lag (m)'); ax.set_ylabel('semivariance (m$^2$)'); ax.set_title('Spatial continuity: variograms', fontsize=10); ax.legend(fontsize=8)

    # metrics block
    ax = axes[2, 1]; ax.axis('off')
    j = jsd(gv, pv)
    txt = (f"Patch {r['pid']}   (ZNCC {r['zncc_saved']:.3f}, {name} of validation set)\n\n"
           f"{'':<18}{'LiDAR':>10}{'Prediction':>12}\n"
           f"{'std (m)':<18}{summary['LiDAR']['std']:>10.4f}{summary['Prediction']['std']:>12.4f}\n"
           f"{'eff. range (m)':<18}{summary['LiDAR']['eff_range']:>10.1f}{summary['Prediction']['eff_range']:>12.1f}\n"
           f"{'sill (m^2)':<18}{summary['LiDAR']['sill']:>10.5f}{summary['Prediction']['sill']:>12.5f}\n\n"
           f"Jensen-Shannon divergence (GT vs pred): {j:.4f}\n"
           f"std ratio pred/GT: {summary['Prediction']['std']/summary['LiDAR']['std']:.3f}")
    ax.text(0.02, 0.95, txt, family='monospace', fontsize=10, va='top', transform=ax.transAxes,
            bbox=dict(facecolor='#F7F8F9', edgecolor='#9AA5AB'))
    plt.tight_layout()
    out = OUTPUT_DIR / f'fig_morph_stat_patch_{name}.png'
    plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white'); print('Saved:', out)
    plt.show()
    return dict(patch=r['pid'], zncc=r['zncc_saved'], jsd=j, **{f'{k}_{m}': v for k, d in summary.items() for m, v in d.items()})

all_stats = {name: analyse(name, r) for name, r in results.items()}
json.dump(all_stats, (OUTPUT_DIR / 'fig_morph_stat_patch_stats.json').open('w'), indent=2)
print(json.dumps(all_stats, indent=2))